# ANSER Brain — fine-tune v3 trên Colab

**GPU đích: L4 22.5GB (Colab Pro).** Notebook chạy tuần tự từ trên xuống.

| Giai đoạn | Nội dung | Thời gian ước tính |
|---|---|---|
| 1 | Cài đặt, sinh dữ liệu, preflight | ~25 phút (phần lớn là gọi DeepSeek API) |
| 2 | Train QLoRA + gộp + lượng tử hoá AWQ | ~2–3 giờ |
| — | **RESTART RUNTIME** (vLLM xung đột thư viện train) | 1 phút |
| 3 | Benchmark model gốc vs model đã train | ~40 phút |

**Nguyên tắc:** không có cell nào tự ý ghi đè dữ liệu cũ. Mọi bước sinh dữ liệu
đều *resume* — chạy lại là tiếp tục chỗ dở, không làm lại từ đầu.

---
## Giai đoạn 1 — Chuẩn bị

### 1.1 Kiểm tra GPU
Nếu không thấy L4: *Runtime → Change runtime type → L4 GPU*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch

print("bf16 hỗ trợ:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### 1.2 Gắn Drive và lấy mã nguồn

Mã nguồn lấy thẳng từ thư mục bạn đã đẩy lên Drive
(`ANSER_AI_Logistics/AI_ANSER`) — **không cần clone GitHub, không cần token**.

Cell tự dò tìm thư mục repo trong Drive bằng file mốc, nên bạn đổi tên hay di
chuyển thư mục vẫn chạy được.

**Vì sao chép sang `/content` chứ không chạy thẳng trên Drive:** Drive gắn qua
FUSE, mỗi thao tác file mất hàng chục ms. Python import vài trăm file nhỏ và
bước train ghi checkpoint liên tục sẽ chậm tới mức không dùng được. Bản chép ở
`/content` chạy tốc độ đĩa thật; mọi thứ CẦN GIỮ (dữ liệu sinh, model, báo cáo)
vẫn ghi thẳng ra Drive ở các cell sau.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
WORK_DIR   = Path('/content/ANSER_AI')      # nơi thực sự chạy (đĩa nhanh)

# Dò thư mục repo: ưu tiên đường dẫn đã biết, sau đó quét theo file mốc.
MARKERS = ('offline_training/build_dataset_v3.py', 'src/core/prompts.py')

def is_repo(p: Path) -> bool:
    return all((p / m).exists() for m in MARKERS)

candidates = [DRIVE_ROOT / 'ANSER_AI_Logistics' / 'AI_ANSER']
if not is_repo(candidates[0]):
    for marker in DRIVE_ROOT.rglob('offline_training/build_dataset_v3.py'):
        root = marker.parent.parent
        if is_repo(root):
            candidates.insert(0, root)
            break

SRC_DIR = next((c for c in candidates if is_repo(c)), None)
if SRC_DIR is None:
    raise SystemExit(
        'Không tìm thấy repo trong Drive.\n'
        'Kiểm tra thư mục đã đẩy lên có chứa offline_training/ và src/ hay chưa.'
    )
print(f'Nguồn  : {SRC_DIR}')

# Chép sang /content, BỎ QUA rác nặng: _backups (234MB), .git, cache, model cũ
SKIP = {'_backups', '.git', '__pycache__', '.pytest_cache', '.ruff_cache',
        '.mypy_cache', 'ANSER', 'checkpoints'}
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(SRC_DIR, WORK_DIR,
                ignore=shutil.ignore_patterns(*SKIP), dirs_exist_ok=True)

os.chdir(WORK_DIR)
n_files = sum(1 for _ in WORK_DIR.rglob('*') if _.is_file())
size_mb = sum(f.stat().st_size for f in WORK_DIR.rglob('*') if f.is_file()) / 1e6
print(f'Làm việc: {WORK_DIR}  ({n_files} file, {size_mb:.0f} MB)')

# Dữ liệu sinh ra ghi THẲNG lên Drive — tốn tiền API, không được mất khi hết phiên
GEN_DIR = DRIVE_ROOT / 'ANSER_AI_Logistics' / 'generated'
GEN_DIR.mkdir(parents=True, exist_ok=True)
os.environ['ANSER_GENERATED_DIR'] = str(GEN_DIR)
print(f'Dữ liệu sinh: {GEN_DIR}  (giữ qua các phiên, chạy lại là resume)')

### 1.3 Cài thư viện train

Version **ghim cứng** trong `requirements_training.txt`. Đừng cài bản mới nhất:
`train_v2.py` cũ chết chính vì API thư viện trôi theo thời gian.

Colab có thể báo xung đột với gói cài sẵn — bỏ qua được, miễn là import ở cell
sau chạy trót lọt.

In [ ]:
!pip install -q -r offline_training/requirements_training.txt
print("\n--- kiểm tra import ---")
import accelerate
import bitsandbytes
import datasets
import peft
import transformers

for m in (transformers, peft, datasets, accelerate, bitsandbytes):
    print(f"{m.__name__:16s} {m.__version__}")

### 1.4 Kiểm tra dung lượng Drive

Model AWQ đầu ra ~5.5GB lưu vào `ANSER_AI_Logistics/anser-v3-awq`.
Bản gộp fp16 (~16GB) chỉ nằm ở `/content` — **không** đẩy lên Drive.

Thiếu chỗ thì xoá: `_backups` (234MB, bản sao lưu cục bộ không cần trên Drive),
`anser-qwen-lora` và `anser-qwen-distill-awq` (model **v1**, đã bị v2 thay thế).

In [ ]:
import shutil
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/ANSER_AI_Logistics')
DATA_DIR    = Path('/content/drive/MyDrive/ANSER_data')   # nơi để model, có từ trước

free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print(f'Drive còn trống: {free_gb:5.1f} GB   (cần >= 8 GB)\n')

for base in (PROJECT_DIR, DATA_DIR):
    if not base.exists():
        print(f'{base.name}/  (chưa có)')
        continue
    total = sum(f.stat().st_size for f in base.rglob('*') if f.is_file()) / 1e9
    print(f'{base.name}/  — {total:.1f} GB')
    for d in sorted(p for p in base.iterdir() if p.is_dir()):
        size = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1e9
        flag = '   <- xoá được (bản sao lưu cục bộ)' if d.name == '_backups' else ''
        flag = '   <- model v1, đã bị v2 thay thế' if d.name.startswith('anser-qwen') else flag
        print(f'    {d.name:28s} {size:5.2f} GB{flag}')

if free_gb < 8:
    print('\n*** THIẾU CHỖ — xoá bớt trước khi train ***')

### 1.5 Khoá API DeepSeek

**Không dán khoá thẳng vào notebook** (AGENTS.md R2b). Thêm vào Colab Secrets:
biểu tượng 🔑 bên trái → *Add new secret* → tên `DEEPSEEK_API_KEY` → bật
*Notebook access*.

Chi phí ước tính cho toàn bộ bước sinh dữ liệu: **dưới 1 USD**.

In [ ]:
import os

from google.colab import userdata

try:
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
    print("✓ Đã nạp DEEPSEEK_API_KEY từ Colab Secrets (không in ra giá trị)")
except Exception as exc:
    print(f"✗ Chưa lấy được khoá: {exc}")
    print("  Bước 1.6b và 1.6c sẽ không chạy được. Xem hướng dẫn ở cell trên.")

### 1.6 Sinh dữ liệu

Năm bước. **a**, **e** không cần mạng; **b**, **c**, **d** gọi DeepSeek.
Cả ba bước gọi API đều *resume* — đứt giữa chừng thì chạy lại chính cell đó,
nó bỏ qua phần đã xong (dữ liệu nằm trên Drive nên không mất tiền lần hai).

| Bước | Nội dung | Nhánh được dạy |
|---|---|---|
| a + b | Trích xuất yêu cầu báo giá, 25% là cặp 2 lượt | `LOGISTICS_EXTRACT` |
| c | Diễn giải số liệu, giải thích xAI, báo cáo từ số engine thật | `DATA` / `EXPLAIN` / `REPORT` |
| d | Vòng agentic: chọn tool, điền tham số, biết hỏi lại khi thiếu | `AGENT` |
| e | Workflow n8n từ template đang chạy thật | `CODER` |

In [ ]:
# a) Ground truth trích xuất (tất định — nhãn đúng tuyệt đối, 25% là cặp 2 lượt)
!python offline_training/make_extraction_seeds.py --n 600 --n-eval 100

In [ ]:
# b) DeepSeek viết tin nhắn cho ground truth có sẵn (~15 phút, resume được)
!python offline_training/reverse_generate.py

In [ ]:
# c) Diễn giải + giải thích (xAI) + báo cáo — số do engine tất định tính,
#    teacher chỉ viết lời (~8 phút)
!python offline_training/make_narration_pairs.py --n 220

In [ ]:
# d) Vòng agentic — câu hỏi CÓ đủ dữ kiện -> gọi tool thật -> trả lời;
#    kèm ca thiếu dữ kiện phải HỎI LẠI thay vì bịa tham số (~10 phút)
!python offline_training/make_agent_traces.py --n 160

In [ ]:
# e) Cặp yêu cầu -> workflow từ template n8n ĐANG CHẠY THẬT
!python offline_training/make_n8n_pairs.py

In [ ]:
# e) Gộp tất cả + lọc + quét secret + ghi vân tay catalog
!python offline_training/build_dataset_v3.py

### 1.7 Preflight — chốt chặn trước khi đốt giờ GPU

Kiểm tra: đủ file, đúng cấu trúc vai, **prompt trong data khớp runtime** (P4),
nhãn hợp lệ theo schema, không có secret, độ dài token vừa ngân sách L4, đủ chỗ Drive.

**Thoát mã 1 = có lỗi chặn. Đừng train khi còn lỗi chặn.**

In [ ]:
!python offline_training/preflight_check.py --tokenizer --gpu L4 \
    --drive /content/drive/MyDrive/ANSER_AI_Logistics

### 2.1 Train QLoRA

Mặc định: Qwen3-8B, LoRA r=32, 2 epoch, seq 8192, lô hiệu dụng 16.
Loss **chỉ tính trên phần trả lời**; chọn checkpoint theo `eval_loss` tốt nhất.

**Liger Kernel là bắt buộc với Qwen3.** Từ vựng 151.936 token khiến hàm loss
của transformers dựng tensor `độ_dài × 151.936` rồi **nhân đôi** khi ép fp32 →
OOM *giữa chừng* khi gặp mẫu dài. Liger gộp `lm_head` + cross-entropy thành một
kernel, không bao giờ dựng nguyên tensor đó.

`PYTORCH_CUDA_ALLOC_CONF` phải đặt **trước khi CUDA khởi tạo** — nên nó nằm ở
cell này, và sau khi restart **đừng chạy lại cell 1.1** (cell đó chạm CUDA).

Muốn chắc không OOM trước khi train thật: bỏ chú thích dòng `EPOCHS = "0.05"`,
chạy ~5 phút, thấy trôi thì chú thích lại.

In [ ]:
# Đặt TRƯỚC mọi thao tác CUDA — chống phân mảnh (lần OOM trước kẹt 2,26 GB)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Cài CÙNG một lệnh: liger kéo theo transformers mới hơn, mà bản đó từ chối
# chạy 4-bit với bitsandbytes cũ. Cài riêng lẻ là gãy ở bước nạp model.
!pip install -q liger-kernel "bitsandbytes>=0.46.1"

os.environ["BASE_MODEL_ID"] = "Qwen/Qwen3-8B"
os.environ["MAX_SEQ_LEN"]   = "8192"      # hạ về "3072" nếu vẫn OOM
os.environ["LORA_R"]        = "32"
os.environ["LR"]            = "1e-4"
os.environ["EPOCHS"]        = "2"
# os.environ["EPOCHS"]      = "0.05"      # <- chạy thử ~5 phút để chắc không OOM
os.environ["OUT_DIR"]       = "/content/checkpoints/anser-v3"
os.environ["LORA_DIR"]      = "/content/checkpoints/anser-v3-lora"

exec(open('/content/ANSER_AI/offline_training/train_v3.py').read())

### 2.2 Sao lưu LoRA lên Drive ngay

LoRA chỉ ~300MB. Sao lưu trước khi làm bước nặng tiếp theo — nếu quantize hỏng
hoặc mất phiên, không phải train lại.

In [ ]:
!cp -r /content/checkpoints/anser-v3-lora /content/drive/MyDrive/ANSER_AI_Logistics/
!du -sh /content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-lora

### 2.3 Gộp LoRA vào weights đầy đủ

LoRA train trên nền 4-bit nhưng phải gộp vào bản fp16/bf16. Bản gộp ~16GB nằm ở
`/content`, **không** lên Drive.

> **Cell này tự tìm LoRA.** `Runtime > Restart` xoá sạch `/content`, nên bản trên
> Drive mới là bản sống sót — cell ưu tiên nó. Chạy được sau restart mà không
> cần chạy lại 2.1.

In [ ]:
# 2.3 — Gop LoRA. Cell nay TU DAT duong dan, khong dua vao cell 2.1.
#
# Runtime > Restart xoa sach /content, nen LORA_DIR mac dinh
# (/content/checkpoints/anser-v3-lora) bien mat sau moi lan restart. Ban tren
# Drive moi la ban song sot -> uu tien no.
import os
from pathlib import Path

CANDIDATES = [
    "/content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-lora",
    "/content/checkpoints/anser-v3-lora",
]
lora = next((p for p in CANDIDATES if Path(p, "adapter_config.json").exists()), None)
assert lora, (
    "Khong thay LoRA o bat ky dau nao:\n  " + "\n  ".join(CANDIDATES) +
    "\n\nNeu Colab da thu hoi may thi phai train lai tu 2.1."
)
print(f"Dung LoRA tai: {lora}")

os.environ["BASE_MODEL_ID"] = "Qwen/Qwen3-8B"
os.environ["LORA_DIR"]      = lora
os.environ["MERGED_DIR"]    = "/content/checkpoints/anser-v3-merged"

!python offline_training/merge_and_quantize.py --stage merge

### 2.4 Lượng tử hoá AWQ 4-bit

Dữ liệu calibration lấy từ chính `train_v3.jsonl` (in-domain — chuẩn hơn
wikitext mặc định cho tiếng Việt + JSON n8n).

**Nếu cell này lỗi cài đặt `autoawq`:** khả năng cao là xung đột version
`transformers`. Cách xử lý: *Runtime → Restart*, chạy lại cell 1.2 (lấy mã),
rồi chạy thẳng cell này (không cần cài lại thư viện train).

In [ ]:
import os
os.environ['AWQ_DIR'] = '/content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq'

!pip install -q autoawq==0.2.9
!python offline_training/merge_and_quantize.py --stage quant

In [ ]:
!du -sh /content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq
!ls -la /content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq | head

---
## ⚠️ RESTART RUNTIME TẠI ĐÂY

vLLM cần bộ thư viện khác với bộ dùng để train. **Runtime → Restart runtime**,
rồi chạy lại **đúng cell 1.2** (gắn Drive + lấy mã nguồn + đặt
`ANSER_GENERATED_DIR`), sau đó tiếp tục từ 3.1.

Không mất gì: dữ liệu sinh nằm trên Drive, model đã lưu Drive, bản gộp fp16 vẫn
ở `/content` (chỉ mất nếu Colab thu hồi máy, khi đó chạy lại từ 2.3).

---
## Giai đoạn 3 — Đo lường

### 3.1 Cài vLLM

In [ ]:
!pip install -q vllm==0.8.5
import vllm; print("vLLM", vllm.__version__)

### 3.2 Baseline — model GỐC chưa fine-tune

**Chạy cell này trước khi xem điểm model mới.** Không có baseline thì con số
"85% đúng" chẳng nói lên điều gì: có thể model gốc đã đạt 84%, tức fine-tune
gần như vô ích; cũng có thể gốc chỉ 40%, tức cải thiện rất lớn.

`--no-gate` = chỉ đo, không chặn. Mất ~20 phút (phần lớn là tải model 16GB).

In [ ]:
!python offline_training/benchmark_v3.py --model Qwen/Qwen3-8B --no-gate \
    2>&1 | tee /content/baseline_report.txt

### 3.3 Model đã fine-tune

Không có `--no-gate` → **thoát mã 1 nếu dưới ngưỡng**. Ngưỡng mặc định:

| Chỉ số | Ngưỡng | Ý nghĩa |
|---|---|---|
| `EXTRACT_FIELD_MIN` | 0.85 | độ chính xác trung bình các trường trích xuất |
| `EXTRACT_READY_MIN` | 0.80 | tỷ lệ đủ 3 trường bắt buộc để tính được báo giá |
| `EXTRACT_FOLLOWUP_MIN` | 0.70 | **riêng câu nối tiếp** — đo khả năng kế thừa ngữ cảnh |
| `N8N_VALID_MIN` | 0.90 | workflow sinh ra qua được validator |
| `NARR_MIN` | 0.90 | không bịa số, không lộ biên lợi nhuận |
| `AGENT_TOOL_MIN` | 0.85 | chọn đúng công cụ cho câu hỏi có đủ dữ kiện |
| `AGENT_ASKBACK_MIN` | 0.75 | **thiếu dữ kiện thì hỏi lại**, không gọi tool với tham số bịa |

Hai chỉ số cuối là mới. `AGENT_ASKBACK_MIN` quan trọng hơn `AGENT_TOOL_MIN`:
chọn nhầm tool thì kết quả sai lộ ra ngay, còn gọi tool với tham số **bịa** thì
ra một con số trông hợp lý mà không ai biết là sai.

In [ ]:
!python offline_training/benchmark_v3.py \
    --model /content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq \
    2>&1 | tee /content/tuned_report.txt

### 3.4 So sánh

Đọc theo thứ tự ưu tiên:

1. **`sẵn sàng báo giá`** — chỉ số sát nghiệp vụ nhất: bao nhiêu % tin nhắn
   khách gửi ra được báo giá mà không phải hỏi lại.
2. **`câu nối tiếp`** — thấp hơn hẳn câu một lượt nghĩa là hội thoại nhiều lượt
   chưa dùng được, dù điểm chung có đẹp.
3. **`false_fill`** — model **đoán bừa** trường mà tin nhắn không nêu. Đây là
   lỗi nguy hiểm nhất: nó phá nhánh hỏi-lại và tạo báo giá sai tuyến trong im lặng.
4. **`n8n hợp lệ`** — dưới 90% thì nhánh tạo quy trình chưa giao cho khách được.

In [ ]:
print("=" * 62, "\n  BASELINE (Qwen3-8B gốc)\n", "=" * 62)
print(open('/content/baseline_report.txt').read()[-2500:])
print("=" * 62, "\n  ĐÃ FINE-TUNE (anser-v3-awq)\n", "=" * 62)
print(open('/content/tuned_report.txt').read()[-2500:])

In [ ]:
# Lưu cả hai báo cáo lên Drive để đối chiếu với lần train sau
!cp /content/baseline_report.txt /content/tuned_report.txt \
    /content/drive/MyDrive/ANSER_AI_Logistics/
!ls -la /content/drive/MyDrive/ANSER_AI_Logistics/*.txt

---
## Xong. Dùng model như thế nào

Trên máy chạy Brain (không phải Colab — xem ARCHITECTURE §4.4: Colab **cấm**
phục vụ traffic thật):

```bash
export TEXT_MODEL_ID=/đường/dẫn/tới/anser-v3-awq
```

`config.py` đọc biến này; `vllm_config` đã đặt sẵn `quantization="awq"`.

### Nếu điểm chưa đạt ngưỡng

| Triệu chứng | Hướng xử lý |
|---|---|
| `false_fill` cao (đoán bừa trường thiếu) | Tăng tỷ lệ seed thiếu trường: sửa `MISSING_PATTERNS` trong `make_extraction_seeds.py` |
| Câu nối tiếp kém hơn hẳn câu một lượt | Tăng `FOLLOWUP_RATIO` (mặc định 0.25) rồi sinh lại dữ liệu |
| n8n hợp lệ thấp | Kiểm tra `N8N_TEMPLATES_DIR` lúc serve có trỏ đúng thư mục lúc train không — `preflight_check.py` so vân tay catalog |
| Mọi chỉ số đều kém hơn baseline | Overfit: giảm `EPOCHS` về 1, hoặc giảm `LORA_R` về 16 |
| Điểm gần bằng baseline | Fine-tune đóng góp ít — guided decoding đã gánh phần lớn. Cân nhắc dùng thẳng model gốc, để dành tiền cho hạ tầng |